In [1]:
# Install dependencies (run once per environment)
! pip install -U "transformers" "datasets" "evaluate" "scikit-learn" "wandb" "optuna" "accelerate"

In [2]:
import pandas as pd
import numpy as np
import torch
import optuna
import json
import ast
import wandb
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from pathlib import Path

In [3]:
from google.colab import drive
drive.mount("/content/drive")
data_dir = Path("/content/drive/MyDrive/ContentID/data")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- Configuration ---
MODEL_NAME = "google/gemma-3-1b-it"  # Gemma 3 1B instruction-tuned (no quantization)
INPUT_COL = "messages"
TARGET_COL = "label"
MAX_LENGTH = 1024
N_TRIALS = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WANDB_PROJECT = "content-classifier-gemma"
MODEL_DIR = "safety-classifier-gemma"
BEST_MODEL_DIR = "./best_gemma_classifier"
HF_TOKEN = os.getenv("HF_TOKEN", '')

In [5]:
! pip install huggingface_hub

In [6]:
from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
# Read data from data_dir and load the csv files into pandas dataframe
train_df = pd.read_csv(data_dir / "train" / "train_sampled.csv")
val_df = pd.read_csv(data_dir / "train" / "val_sampled.csv")

test_df = pd.read_csv(data_dir / "test" / "test_dataset.csv")

In [ ]:
from torch import nn

class BCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = nn.BCEWithLogitsLoss()
        loss = loss_fct(logits.view(-1), labels.float().view(-1))
        
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    """
    Calculates AUPR, ROC-AUC, and FPR @ Specific Recall levels.
    """
    logits, labels = eval_pred
    
    # Use sigmoid for binary classification with num_labels=1
    probs = torch.sigmoid(torch.tensor(logits)).numpy().flatten()

    # 1. AUPR (Area Under Precision-Recall Curve)
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    aupr = auc(recall, precision)

    # 2. ROC-AUC
    roc_auc = roc_auc_score(labels, probs)

    # 3. FPR at X% Recall
    def get_fpr_at_recall(target_recall):
        idx = np.where(recall >= target_recall)[0]
        if len(idx) == 0: return 1.0
        thr = thresholds[min(idx[0], len(thresholds)-1)]
        preds = (probs >= thr).astype(int)

        fp = np.sum((preds == 1) & (labels == 0))
        tn = np.sum((preds == 0) & (labels == 0))
        return fp / (fp + tn) if (fp + tn) > 0 else 0.0

    fpr_90 = get_fpr_at_recall(0.90)
    fpr_95 = get_fpr_at_recall(0.95)

    return {
        "aupr": aupr,
        "roc_auc": roc_auc,
        "fpr_at_90_recall": fpr_90,
        "fpr_at_95_recall": fpr_95
    }

def parse_messages(text):
    """
    Parse the messages column from a string representation of a list of dicts
    into an actual list of dicts (OpenAI chat format).
    """
    try:
        msgs = ast.literal_eval(text)
        if isinstance(msgs, list):
            return msgs
    except:
        pass
    # Fallback: wrap raw text as a single user message
    return [{"role": "user", "content": str(text)}]

def get_prepared_datasets(train_df, val_df):
    """Helper to load and tokenize data using Gemma's chat template."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Gemma doesn't have a pad_token by default; set it to eos_token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Provide Gemma's official chat template to the tokenizer 
    # to format multi-turn conversations cleanly before tokenization
    tokenizer.chat_template = "{% if messages[0]['role'] == 'system' %}{{ messages[0]['content'] }}{% set loop_messages = messages[1:] %}{% else %}{% set loop_messages = messages %}{% endif %}{% for message in loop_messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if message['role'] == 'user' %}{{ '<start_of_turn>user\n' + message['content'] + '<end_of_turn>\n' }}{% elif message['role'] == 'assistant' %}{{ '<start_of_turn>model\n' + message['content'] + '<end_of_turn>\n' }}{% endif %}{% endfor %}"

    def tokenize_function(examples):
        # Apply Gemma's chat template to format List[dict] messages into a single string
        texts = [
            tokenizer.apply_chat_template(
                parse_messages(msg), tokenize=False, add_generation_prompt=False
            )
            for msg in examples[INPUT_COL]
        ]
        return tokenizer(texts, truncation=True, padding=False, max_length=MAX_LENGTH)

    train_ds = Dataset.from_pandas(train_df[[INPUT_COL, TARGET_COL]]).map(tokenize_function, batched=True, remove_columns=[INPUT_COL])
    val_ds = Dataset.from_pandas(val_df[[INPUT_COL, TARGET_COL]]).map(tokenize_function, batched=True, remove_columns=[INPUT_COL])

    train_ds = train_ds.rename_column(TARGET_COL, "labels")
    val_ds = val_ds.rename_column(TARGET_COL, "labels")

    return train_ds, val_ds, tokenizer

def objective(trial):
    """Optuna objective function for Bayesian Optimization."""

    run = wandb.init(
        project=WANDB_PROJECT,
        group="bayesian-optimization",
        job_type="optimization",
        name=f"trial_{trial.number}",
        reinit=True
    )

    train_ds, val_ds, tokenizer = get_prepared_datasets(train_df, val_df)

    # Hyperparameters to optimize
    learning_rate = trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    warmup_steps = trial.suggest_int("warmup_steps", 0, 500)

    # Gradient accumulation to match effective batch size while keeping memory usage in check
    train_batch = max(2, batch_size // 4)

    training_args = TrainingArguments(
        output_dir=f"./temp_gemma_results_trial_{trial.number}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch,
        per_device_eval_batch_size=train_batch,
        gradient_accumulation_steps=batch_size // train_batch,
        num_train_epochs=3,
        weight_decay=weight_decay,
        warmup_steps=warmup_steps,
        fp16=False,
        bf16=torch.cuda.is_available(),  # Use bf16 on supported GPUs (Ampere+)
        max_grad_norm=1.0,
        logging_steps=10,
        report_to="wandb",
        load_best_model_at_end=False,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )
    model.config.pad_token_id = tokenizer.pad_token_id

    # Use custom BCETrainer
    trainer = BCETrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    try:
        trainer.train()
        eval_results = trainer.evaluate()
        wandb.log(eval_results)
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        wandb.finish()
        raise e

    run.finish()

    return eval_results["eval_aupr"]

In [9]:
print(f"Starting Bayesian Optimization with Optuna on {DEVICE}...")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)
print("\nBest Trial Found:")
best_params = study.best_trial.params
print(f"  AUPR: {study.best_trial.value}")
print(f"  Params: {best_params}")

[I 2026-03-03 01:33:42,176] A new study created in memory with name: no-name-7957e508-3a9f-430e-9a21-70e8aac819bf


Starting Bayesian Optimization with Optuna on cuda...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: vjayram-enag (vijayram-enag-ucr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

...

Gemma3TextForSequenceClassification LOAD REPORT from: google/gemma-3-1b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Aupr,Roc Auc,Fpr At 90 Recall,Fpr At 95 Recall
1,2.373893,0.575585,0.801375,0.796385,1.000000,1.000000
2,1.613558,0.565881,0.810244,0.806442,1.000000,1.000000
3,2.065235,0.572709,0.810492,0.806013,1.000000,1.000000


epoch,▁
eval/aupr,▁███
eval/fpr_at_90_recall,▁▁▁▁
eval/fpr_at_95_recall,▁▁▁▁
eval/loss,█▁▆▆
eval/roc_auc,▁███
eval/runtime,▁▁█▇
eval/samples_per_second,██▁▂
eval/steps_per_second,██▁▂
eval_aupr,▁
+12,...


[I 2026-03-03 04:32:32,908] Trial 2 finished with value: 0.8104915570013393 and parameters: {'learning_rate': 1.8771700573896603e-06, 'weight_decay': 0.004852119118972676, 'batch_size': 8, 'warmup_steps': 321}. Best is trial 1 with value: 0.9464491155474754.



Best Trial Found:
  AUPR: 0.9464491155474754
  Params: {'learning_rate': 1.2534106923321682e-05, 'weight_decay': 0.058894635975275424, 'batch_size': 16, 'warmup_steps': 466}


In [10]:
# --- FINAL TRAINING & SAVING BEST MODEL ---
print("\nTraining final model with best parameters...")

final_run = wandb.init(
    project=WANDB_PROJECT,
    name="final_best_model",
    job_type="final_train"
)

train_ds, val_ds, tokenizer = get_prepared_datasets(train_df=train_df, val_df=val_df)

final_training_args = TrainingArguments(
    output_dir=BEST_MODEL_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_params["learning_rate"],
    per_device_train_batch_size=best_params["batch_size"],
    per_device_eval_batch_size=best_params["batch_size"],
    num_train_epochs=5,
    weight_decay=best_params["weight_decay"],
    warmup_steps=best_params["warmup_steps"],
    bf16=torch.cuda.is_available(),
    report_to="wandb",
    load_best_model_at_end=True,
    metric_for_best_model="aupr",
    greater_is_better=True,
)

final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
final_model.config.pad_token_id = tokenizer.pad_token_id

final_trainer = BCETrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
final_trainer.train()


Training final model with best parameters...


Map:   0%|          | 0/14148 [00:00<?, ? examples/s]

Map:   0%|          | 0/1566 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4425, training_loss=0.21187181849937656, metrics={'train_runtime': 2229.2674, 'train_samples_per_second': 31.732, 'train_steps_per_second': 1.985, 'total_flos': 1.6231014535316275e+17, 'train_loss': 0.21187181849937656, 'epoch': 5.0})

In [11]:
# Evaluate on test set
test_ds = Dataset.from_pandas(test_df[[INPUT_COL, TARGET_COL]])
test_ds = test_ds.rename_column(TARGET_COL, "labels")

def tokenize_function(examples):
    texts = [
        tokenizer.apply_chat_template(
            parse_messages(msg), tokenize=False, add_generation_prompt=False
        )
        for msg in examples[INPUT_COL]
    ]
    return tokenizer(texts, truncation=True, padding=False, max_length=MAX_LENGTH)

test_ds = test_ds.map(tokenize_function, batched=True, remove_columns=[INPUT_COL])

print("Evaluating on test set...")
test_metrics = final_trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")

print("\nTest Set Metrics:")
print(json.dumps(test_metrics, indent=2))

wandb.log(test_metrics)

Map:   0%|          | 0/3780 [00:00<?, ? examples/s]

Evaluating on test set...



Test Set Metrics:
{
  "test_loss": 0.3388453423976898,
  "test_aupr": 0.9812763803340158,
  "test_roc_auc": 0.9799664063156127,
  "test_fpr_at_90_recall": 1.0,
  "test_fpr_at_95_recall": 1.0,
  "test_runtime": 18.3387,
  "test_samples_per_second": 206.122,
  "test_steps_per_second": 12.924,
  "epoch": 5.0
}


In [12]:
# Log final metrics and config to wandb
final_eval = final_trainer.evaluate()
wandb.log(final_eval)
wandb.config.update(best_params)

final_run.finish()
print("Optimization and training complete.")

epoch,▁▁
eval/aupr,▁▇█▆▅█
eval/fpr_at_90_recall,▁▁▁▁▁▁
eval/fpr_at_95_recall,▁▁▁▁▁▁
eval/loss,▂▁▃██▃
eval/roc_auc,▁▇█▆▆█
eval/runtime,█▅▁▅▄▂
eval/samples_per_second,▁▄█▄▅▇
eval/steps_per_second,▁▄█▄▅▇
eval_aupr,▁
+28,...


Optimization and training complete.


In [13]:
# Save the model to Hugging Face Hub
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN)

    hub_model_id = "content-classifier-gemma"

    print(f"Pushing model to Hugging Face Hub: {hub_model_id}")
    final_trainer.model.push_to_hub(hub_model_id)
    tokenizer.push_to_hub(hub_model_id)
    print("Successfully pushed to Hub.")

Pushing model to Hugging Face Hub: content-classifier-gemma


README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpw3t077nl/tokenizer.json:  24%|##3       | 7.98MB / 33.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Successfully pushed to Hub.
